Botnet traffic

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [2]:
print(" Loading the Friday Morning (Botnet) Dataset")
file_path = "MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

 Loading the Friday Morning (Botnet) Dataset


In [ ]:
print("\n Traffic Distribution (Before Cleaning)")
#BENIGN and Bot traffic
print(df['Label'].value_counts())


--- Traffic Distribution (Before Cleaning) ---
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64


In [4]:
print("\n Cleaning and Encoding Data")
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
print(df_clean['Label'].value_counts())


 Cleaning and Encoding Data
Label
0    188955
1      1956
Name: count, dtype: int64


In [5]:
print("\n Splitting Data (Before SMOTE)")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


 Splitting Data (Before SMOTE)


In [6]:
print("\n Applying SMOTE to balance the Training Data")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


 Applying SMOTE to balance the Training Data


In [7]:
print("\n Scaling the Data")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)


 Scaling the Data


In [8]:
print("\n Multi-Model Cycling")
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []


 Multi-Model Cycling


In [9]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [11]:
print("\n Friday Morning (BOTNET) Leaderboard")
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
display(results_df)


 Friday Morning (BOTNET) Leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
3,LightGBM,99.9188,99.7608,0.9642,6.79
0,Decision Tree,99.9136,99.2823,0.9618,5.75
2,XGBoost,99.8821,99.7608,0.9488,6.58
1,Random Forest,99.5338,99.7608,0.8241,10.63
5,MLP (Neural Net),98.4260,99.0431,0.5794,238.71
4,Logistic Regression,95.4482,98.8038,0.3222,10.84


For this dataset, we know that we have a big imbalance (99% normal traffic, and 1% attack), we have used smote earlier and we will judge the end result by the F1 score while also paying attention to the recall so it stays high.

The LightGB: All the important things are present, it has a high recall catch rate, the highest f1 score and the time in which it performed is also short. Being a Gradient Boosting algorithm means that it will learn from it's mistakes. It keeps looking for errors, so it is really good at actually finding that 1& of attacks without getting confused by the 99% of normal traffic that exists. If we are to look at XGBoost, which is also a Gradient boosting algorithm, we can notice that LightGBM has been quicker and more accurate in F!. This happens because is uses a certain technique called "histogram based binning", which means that the code will group continuous data in discrete buckets. This stops it from being fixated on small things that don't really matter

Decision Tree: The fastest out of them, and also not too bad in comparison with the other models based on f1. But it is still considered greedy because of how it works. Compared to the one i mentioned before, Decision tree will create a single 'tree', which is a massive flowchart of no and yes questions. Being a single tree used, it means that it will be very fast , but because they make decisions based on the present moment they are at, it means that decision tree can't see the final outcome, creating a slightly imperfect rule set. That's why it gave false positives

Random Forest: unfortunately, this one suffered because of SMOTE. So, as always, i set it's n_estimators at 50 for it to create its 50 separate decision trees making them look over normal traffic and attacks. But because smote has generated so many synthetic botnet packets, some of the decision trees memorized the fake data too well. So when it came to testing the real data, the trees that learned too well got too suspicious and flagged it as an attack. So even thought the recall was amazing (catching many attacks), the f1 was bad, because it also flagged normal traffic when it shouldn't have.

Logistic regression: It had the same problem like in other cases because it is a linear model. It tried to draw a single mathematical straight line to separate Normal from Attack, but network traffic is too complex and non-linear (like inter-arrival times, packet flags, and byte counts). Because it couldn't find a straight line, especially in our SMOTEd data, it failed, by it i mean that it started to flag everything as Botnet, that's why it has a high recall with a very low F1 score

MLP: Like always, it did a bad job and it took a very long time, showing again proof of why using the wrong tool will give you the wrong result. MLP which is Deep Learning is more used for images, text and even audios and it is extremely bad when it comes to tabular data. It again used all the iterations i gave it and didn't have enough time to actually to create a good mathematical solution, which resulted in its terrible precision. It also took a long time because it was trying to calculate too many things in a detailed way.